# Conversion Rate - Statistical Analysis

## 1. Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from google.cloud import bigquery

## 2. BigQuery Connection

In [2]:
# Connect to BigQuery using local Google Cloud credentials.
client = bigquery.Client(project="ga4-conversion-analytics")

# Execute a test query.
query = "SELECT 1 AS connection_ok"
result = client.query(query).result()

# Retrieve the first row from the query results.
row = next(iter(result))

print(f"BigQuery connection: {row.connection_ok}")

BigQuery connection: 1


## 3. Session Data

In [3]:
from pathlib import Path

# Access the SQL file in the sibling directory.
sql_path = Path("../sql/06_statistical_export.sql")

# Read and execute the SQL query.
query = sql_path.read_text(encoding="utf-8")
df = client.query(query).result().to_dataframe(create_bqstorage_client=False)

# Check the loaded data.
print(df.shape)
print(df["device_category"].value_counts(dropna=False))

(360129, 6)
device_category
desktop    208942
mobile     143185
tablet       8002
Name: count, dtype: int64


### 3.1 Data Validation

In [4]:
# Check whether each row represents a unique session
unique_sessions = df[["user_pseudo_id", "ga_session_id"]].drop_duplicates().shape[0]

# Check missing values across all columns
missing_values = df.isna().sum()

# Summarize the dataset
print(f"Total sessions: {len(df):,}")
print(f"Unique sessions: {unique_sessions:,}")
print(f"Purchase sessions: {df['has_purchase'].sum():,}")
print(f"Date range: {df['session_date'].min()} to {df['session_date'].max()}")
print("\nMissing values:")
print(missing_values)

Total sessions: 360,129
Unique sessions: 360,129
Purchase sessions: 4,848
Date range: 2020-11-01 to 2021-01-31

Missing values:
user_pseudo_id     0
ga_session_id      0
session_date       0
device_category    0
has_search         0
has_purchase       0
dtype: int64
